In [1]:
import requests
import pandas as pd
import json
from dotenv import load_dotenv
import os
import time
from datetime import datetime

In [2]:
load_dotenv()
API_KEY = os.getenv("RAWG_API_KEY")
base_url = "https://api.rawg.io/api"

In [3]:
game_data = []
for i in range(1,51):
    url = f"{base_url}/games?key={API_KEY}&dates=2015-01-01,2025-12-31&page={i}"
    response = requests.get(url)
    game_data.extend(response.json()['results'])
    time.sleep(0.2)
len(game_data)

1000

In [4]:
game_df = pd.DataFrame(game_data)
game_df.head(2)

,slug,name,playtime,platforms,stores,released,tba,background_image,rating,rating_top,...,clip,tags,esrb_rating,user_game,reviews_count,saturated_color,dominant_color,short_screenshots,parent_platforms,genres
0,the-witcher-3-wild-hunt,The Witcher 3: Wild Hunt,43,"[{'platform': {'id': 4, 'name': 'PC', 'slug': ...","[{'store': {'id': 1, 'name': 'Steam', 'slug': ...",2015-05-18,False,https://media.rawg.io/media/games/618/618c2031...,4.64,5,...,None,"[{'id': 31, 'name': 'Singleplayer', 'slug': 's...","{'id': 4, 'name': 'Mature', 'slug': 'mature', ...",None,7278,0f0f0f,0f0f0f,"[{'id': -1, 'image': 'https://media.rawg.io/me...","[{'platform': {'id': 1, 'name': 'PC', 'slug': ...","[{'id': 4, 'name': 'Action', 'slug': 'action'}..."
1,red-dead-redemption-2,Red Dead Redemption 2,21,"[{'platform': {'id': 4, 'name': 'PC', 'slug': ...","[{'store': {'id': 1, 'name': 'Steam', 'slug': ...",2018-10-26,False,https://media.rawg.io/media/games/511/5118aff5...,4.59,5,...,None,"[{'id': 31, 'name': 'Singleplayer', 'slug': 's...","{'id': 4, 'name': 'Mature', 'slug': 'mature', ...",None,5595,0f0f0f,0f0f0f,"[{'id': -1, 'image': 'https://media.rawg.io/me...","[{'platform': {'id': 1, 'name': 'PC', 'slug': ...","[{'id': 4, 'name': 'Action', 'slug': 'action'}]"


In [5]:
game_df = game_df[['name', 'released', 'rating', 'ratings_count', 'metacritic', 'playtime', 'platforms', 'genres', 'tags', 'esrb_rating']]
game_df.head(2)

,name,released,rating,ratings_count,metacritic,playtime,platforms,genres,tags,esrb_rating
0,The Witcher 3: Wild Hunt,2015-05-18,4.64,7159,92.0,43,"[{'platform': {'id': 4, 'name': 'PC', 'slug': ...","[{'id': 4, 'name': 'Action', 'slug': 'action'}...","[{'id': 31, 'name': 'Singleplayer', 'slug': 's...","{'id': 4, 'name': 'Mature', 'slug': 'mature', ..."
1,Red Dead Redemption 2,2018-10-26,4.59,5463,96.0,21,"[{'platform': {'id': 4, 'name': 'PC', 'slug': ...","[{'id': 4, 'name': 'Action', 'slug': 'action'}]","[{'id': 31, 'name': 'Singleplayer', 'slug': 's...","{'id': 4, 'name': 'Mature', 'slug': 'mature', ..."


In [6]:
game_df['release_year'] = pd.to_datetime(game_df['released']).dt.year
game_df = game_df.drop(columns=['released'])
game_df.head(2)

,name,rating,ratings_count,metacritic,playtime,platforms,genres,tags,esrb_rating,release_year
0,The Witcher 3: Wild Hunt,4.64,7159,92.0,43,"[{'platform': {'id': 4, 'name': 'PC', 'slug': ...","[{'id': 4, 'name': 'Action', 'slug': 'action'}...","[{'id': 31, 'name': 'Singleplayer', 'slug': 's...","{'id': 4, 'name': 'Mature', 'slug': 'mature', ...",2015
1,Red Dead Redemption 2,4.59,5463,96.0,21,"[{'platform': {'id': 4, 'name': 'PC', 'slug': ...","[{'id': 4, 'name': 'Action', 'slug': 'action'}]","[{'id': 31, 'name': 'Singleplayer', 'slug': 's...","{'id': 4, 'name': 'Mature', 'slug': 'mature', ...",2018


In [7]:
def categorize_platforms(platforms):
    pc = any(p['platform']['name'] == 'PC' for p in platforms)
    console = any(p['platform']['name'] in ['PlayStation 4', 'PlayStation 5', 'Xbox One', 'Xbox Series S/X', 'Nintendo Switch'] for p in platforms)
    mobile = any(p['platform']['name'] in ['iOS', 'Android'] for p in platforms)
    return pd.Series({'pc': pc, 'console': console, 'mobile': mobile})

game_df[['pc', 'console', 'mobile']] = game_df['platforms'].apply(categorize_platforms)

game_df['platform_count'] = game_df['platforms'].apply(lambda x: len(x))
game_df = game_df.drop(columns=['platforms'])
game_df.head()

,name,rating,ratings_count,metacritic,playtime,genres,tags,esrb_rating,release_year,pc,console,mobile,platform_count
0,The Witcher 3: Wild Hunt,4.64,7159,92.0,43,"[{'id': 4, 'name': 'Action', 'slug': 'action'}...","[{'id': 31, 'name': 'Singleplayer', 'slug': 's...","{'id': 4, 'name': 'Mature', 'slug': 'mature', ...",2015,True,True,False,7
1,Red Dead Redemption 2,4.59,5463,96.0,21,"[{'id': 4, 'name': 'Action', 'slug': 'action'}]","[{'id': 31, 'name': 'Singleplayer', 'slug': 's...","{'id': 4, 'name': 'Mature', 'slug': 'mature', ...",2018,True,True,False,3
2,Life is Strange,4.12,3763,83.0,6,"[{'id': 3, 'name': 'Adventure', 'slug': 'adven...","[{'id': 31, 'name': 'Singleplayer', 'slug': 's...","{'id': 4, 'name': 'Mature', 'slug': 'mature', ...",2015,True,True,True,9
3,Destiny 2,3.52,2666,82.0,6,"[{'id': 2, 'name': 'Shooter', 'slug': 'shooter...","[{'id': 31, 'name': 'Singleplayer', 'slug': 's...","{'id': 3, 'name': 'Teen', 'slug': 'teen', 'nam...",2017,True,True,False,6
4,God of War (2018),4.54,5137,94.0,15,"[{'id': 4, 'name': 'Action', 'slug': 'action'}]","[{'id': 31, 'name': 'Singleplayer', 'slug': 's...","{'id': 4, 'name': 'Mature', 'slug': 'mature', ...",2018,True,True,False,2


In [8]:
def game_player_type(tags):
    has_single = any(tag['name'] == 'Singleplayer' for tag in tags)
    has_multi = any(tag['name'] == 'Multiplayer' for tag in tags)
    
    if has_single and has_multi:
        return 'Both'
    elif has_single:
        return 'Singleplayer'
    elif has_multi:
        return 'Multiplayer'
    else:
        return 'None'

game_df['game_type'] = game_df['tags'].apply(game_player_type)
game_df = game_df.drop(columns=['tags'])
game_df.head()

,name,rating,ratings_count,metacritic,playtime,genres,esrb_rating,release_year,pc,console,mobile,platform_count,game_type
0,The Witcher 3: Wild Hunt,4.64,7159,92.0,43,"[{'id': 4, 'name': 'Action', 'slug': 'action'}...","{'id': 4, 'name': 'Mature', 'slug': 'mature', ...",2015,True,True,False,7,Singleplayer
1,Red Dead Redemption 2,4.59,5463,96.0,21,"[{'id': 4, 'name': 'Action', 'slug': 'action'}]","{'id': 4, 'name': 'Mature', 'slug': 'mature', ...",2018,True,True,False,3,Both
2,Life is Strange,4.12,3763,83.0,6,"[{'id': 3, 'name': 'Adventure', 'slug': 'adven...","{'id': 4, 'name': 'Mature', 'slug': 'mature', ...",2015,True,True,True,9,Singleplayer
3,Destiny 2,3.52,2666,82.0,6,"[{'id': 2, 'name': 'Shooter', 'slug': 'shooter...","{'id': 3, 'name': 'Teen', 'slug': 'teen', 'nam...",2017,True,True,False,6,Both
4,God of War (2018),4.54,5137,94.0,15,"[{'id': 4, 'name': 'Action', 'slug': 'action'}]","{'id': 4, 'name': 'Mature', 'slug': 'mature', ...",2018,True,True,False,2,Singleplayer


In [9]:
game_df['genre_count'] = game_df['genres'].apply(lambda x: len(x))
game_df['genres'] = game_df['genres'].apply(lambda x: [g['name'] for g in x])
game_df.head()

,name,rating,ratings_count,metacritic,playtime,genres,esrb_rating,release_year,pc,console,mobile,platform_count,game_type,genre_count
0,The Witcher 3: Wild Hunt,4.64,7159,92.0,43,"[Action, RPG]","{'id': 4, 'name': 'Mature', 'slug': 'mature', ...",2015,True,True,False,7,Singleplayer,2
1,Red Dead Redemption 2,4.59,5463,96.0,21,[Action],"{'id': 4, 'name': 'Mature', 'slug': 'mature', ...",2018,True,True,False,3,Both,1
2,Life is Strange,4.12,3763,83.0,6,[Adventure],"{'id': 4, 'name': 'Mature', 'slug': 'mature', ...",2015,True,True,True,9,Singleplayer,1
3,Destiny 2,3.52,2666,82.0,6,"[Shooter, Action]","{'id': 3, 'name': 'Teen', 'slug': 'teen', 'nam...",2017,True,True,False,6,Both,2
4,God of War (2018),4.54,5137,94.0,15,[Action],"{'id': 4, 'name': 'Mature', 'slug': 'mature', ...",2018,True,True,False,2,Singleplayer,1


In [10]:
game_df['esrb_rating'] = game_df['esrb_rating'].apply(lambda x: x['name'] if x is not None else None)
game_df.head()

,name,rating,ratings_count,metacritic,playtime,genres,esrb_rating,release_year,pc,console,mobile,platform_count,game_type,genre_count
0,The Witcher 3: Wild Hunt,4.64,7159,92.0,43,"[Action, RPG]",Mature,2015,True,True,False,7,Singleplayer,2
1,Red Dead Redemption 2,4.59,5463,96.0,21,[Action],Mature,2018,True,True,False,3,Both,1
2,Life is Strange,4.12,3763,83.0,6,[Adventure],Mature,2015,True,True,True,9,Singleplayer,1
3,Destiny 2,3.52,2666,82.0,6,"[Shooter, Action]",Teen,2017,True,True,False,6,Both,2
4,God of War (2018),4.54,5137,94.0,15,[Action],Mature,2018,True,True,False,2,Singleplayer,1


In [11]:
game_df['primary_genre'] = game_df['genres'].apply(lambda x: x[0] if x else None)
game_df['genre_count'] = game_df['genres'].apply(lambda x: len(x))
game_df = game_df.drop(columns='genres')
game_df.head()

,name,rating,ratings_count,metacritic,playtime,esrb_rating,release_year,pc,console,mobile,platform_count,game_type,genre_count,primary_genre
0,The Witcher 3: Wild Hunt,4.64,7159,92.0,43,Mature,2015,True,True,False,7,Singleplayer,2,Action
1,Red Dead Redemption 2,4.59,5463,96.0,21,Mature,2018,True,True,False,3,Both,1,Action
2,Life is Strange,4.12,3763,83.0,6,Mature,2015,True,True,True,9,Singleplayer,1,Adventure
3,Destiny 2,3.52,2666,82.0,6,Teen,2017,True,True,False,6,Both,2,Shooter
4,God of War (2018),4.54,5137,94.0,15,Mature,2018,True,True,False,2,Singleplayer,1,Action


In [12]:
game_df = game_df[['name', 'primary_genre', 'rating', 'ratings_count', 'release_year',
                   'metacritic', 'playtime', 'genre_count', 'game_type',
                   'platform_count', 'pc', 'console', 'mobile', 'esrb_rating']]
game_df = game_df.sort_values(by='rating', ascending=False)
game_df = game_df.reset_index(drop=True)
game_df.head()

,name,primary_genre,rating,ratings_count,release_year,metacritic,playtime,genre_count,game_type,platform_count,pc,console,mobile,esrb_rating
0,The Witcher 3 Wild Hunt - Complete Edition,Adventure,4.80,794,2016,92.0,0,2,None,6,True,True,False,Mature
1,Persona 5 Royal,Adventure,4.75,527,2020,94.0,12,2,Singleplayer,6,True,True,False,Mature
2,The Last of Us Part I,Shooter,4.67,548,2022,NaN,13,3,Both,2,True,True,False,Mature
3,Disco Elysium: Final Cut,Adventure,4.66,655,2021,90.0,5,2,None,8,True,True,True,NaN
4,The Witcher 3: Wild Hunt,Action,4.64,7159,2015,92.0,43,2,Singleplayer,7,True,True,False,Mature


In [13]:
game_df.to_csv('game_dataset.csv', index=False)
print("Dataset saved as 'game_dataset.csv'.")

Dataset saved as 'game_dataset.csv'.
